# CREST Phase 1b: Response-Time Per-Head Capture (thinking mode)

**Why**: Phase 1 captured at last prompt token -> top cv_acc 57.8% (noisy). Phase 1b v1 disabled thinking -> 2.8% accuracy (trivial). This version enables thinking.

**Setup**:
- `enable_thinking=True`, `max_new_tokens=2000` (fits most CoT)
- Parse answer from post-`</think>` block, fallback to last `\\boxed{}` anywhere
- 200 Stage B questions, greedy decode
- Hook pre-o_proj, append per-token activations, mean-pool over response span

**Expected**: ~50% accuracy (matches Stage B), top cv_acc 60-65%+ on response mean.

**Budget**: ~3-4h on RTX 6000 Blackwell (200 rollouts x ~1000 tokens actually thought).

## Cell 1 — Setup (same as Phase 1)

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/crest_phase1b'); OUT.mkdir(exist_ok=True)
HF_CREST = 'caiovicentino1/qwen36-crest-cognitive-heads'
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download, HfApi
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
GA_LAYERS = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39]
N_Q_HEADS = 16
HEAD_DIM = 256

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

## Cell 3 — Install append-mode hooks (accumulate across generation steps)

In [ ]:
def find_o_proj(layer):
    for n, m in layer.named_modules():
        if n.endswith('o_proj'):
            return m
    return None

o_proj_modules = {L: find_o_proj(get_layer_module(model, L)) for L in GA_LAYERS}

# Append-mode: during generation, each forward fires hook once
# Input shape: [B, T, n_heads*head_dim]. We take LAST token of each forward.
# Over generation, accumulates [prompt_tail + response_tokens]
gen_head_acts = {L: [] for L in GA_LAYERS}

def make_hook(L):
    def hook(module, inp):
        x = inp[0]
        last = x[:, -1, :].view(-1, N_Q_HEADS, HEAD_DIM)  # [B, 16, 256]
        gen_head_acts[L].append(last[0].float().cpu().numpy())
        return None
    return hook

handles = []
for L in GA_LAYERS:
    h = o_proj_modules[L].register_forward_pre_hook(make_hook(L))
    handles.append(h)
print(f'OK {len(handles)} append-mode hooks installed')

## Cell 4 — Load Stage B questions + balance 250 rollouts

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(
            question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(42); random.shuffle(questions)
N_TARGET = 200
questions = questions[:N_TARGET]
print(f'{len(questions)} unique questions loaded')

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = ("Answer the following multiple-choice question. "
        "Think step by step, then put the final letter in \\boxed{}.\n\n"
        f"Question: {q}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def extract_answer(text):
    # Prefer post-</think> content if present
    post = text.split('</think>')[-1] if '</think>' in text else text
    m = re.search(r'\\boxed\{([A-J])\}', post)
    if m: return m.group(1)
    m = re.search(r'\banswer is\s*\(?([A-J])\)?', post, re.I)
    if m: return m.group(1).upper()
    # Fallback: scan entire text for last boxed
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    letters = re.findall(r'\b([A-J])\b', post[-200:] if post else text[-200:])
    return letters[-1] if letters else None

## Cell 5 — Greedy generate + capture response-span activations

In [ ]:
MAX_NEW = 2000
MAX_PROMPT = 2048

head_acts_list = []
correct_list = []
kept_questions = []
t0 = time.time()

for qi, q in enumerate(tqdm(questions, desc='generate')):
    try:
        p = format_prompt(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.cuda()
        if ids.shape[1] > MAX_PROMPT: continue
        prompt_len = ids.shape[1]

        for L in GA_LAYERS: gen_head_acts[L] = []

        with torch.no_grad():
            out = model.generate(
                ids, max_new_tokens=MAX_NEW, do_sample=False,
                pad_token_id=tok.pad_token_id,
                use_cache=True)

        gen_ids = out[0, prompt_len:].cpu().tolist()
        gen_text = tok.decode(gen_ids, skip_special_tokens=True)
        pred = extract_answer(gen_text)
        correct = (pred == q['gold_letter']) if pred else False

        per_layer_mean = np.zeros((len(GA_LAYERS), N_Q_HEADS, HEAD_DIM), dtype=np.float16)
        ok = True
        for li, L in enumerate(GA_LAYERS):
            acts = gen_head_acts[L]
            if len(acts) < 2:
                ok = False; break
            response_acts = np.stack(acts[1:])
            per_layer_mean[li] = response_acts.mean(axis=0).astype(np.float16)
        if not ok: continue

        head_acts_list.append(per_layer_mean)
        correct_list.append(correct)
        kept_questions.append(dict(question=q['question'], gold=q['gold_letter'],
                                    pred=pred, correct=correct,
                                    n_response=len(gen_head_acts[GA_LAYERS[0]])-1))
        # Periodic progress
        if (qi+1) % 20 == 0:
            nc = sum(correct_list)
            print(f'  [{qi+1}/{len(questions)}] {nc} correct / {len(correct_list)-nc} wrong ({nc/max(1,len(correct_list))*100:.1f}%)')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        continue

for h in handles: h.remove()

head_acts = np.stack(head_acts_list)
correct_mask = np.array(correct_list, dtype=bool)
elapsed = (time.time()-t0)/60
print(f'\nOK {len(head_acts_list)} rollouts in {elapsed:.1f} min')
print(f'shape: {head_acts.shape}, correct rate: {correct_mask.mean()*100:.1f}%')
print(f'mean response len: {np.mean([q["n_response"] for q in kept_questions]):.0f} tokens')

np.savez(OUT / 'head_acts_response.npz', head_acts=head_acts, correct=correct_mask,
         ga_layers=np.array(GA_LAYERS))

## Cell 6 — Train 160 probes on response-span activations

In [ ]:
y = correct_mask.astype(int)
print(f'Class balance: {y.sum()} correct / {(1-y).sum()} wrong')

probe_results = []
print(f'\n{"layer":>5s} {"head":>4s} {"train":>6s} {"cv":>6s}')
for li, L in enumerate(GA_LAYERS):
    for hi in range(N_Q_HEADS):
        X = head_acts[:, li, hi, :].astype(np.float32)
        probe = LogisticRegression(C=0.5, max_iter=2000, random_state=42)
        try:
            cv_scores = cross_val_score(probe, X, y, cv=5, scoring='accuracy')
            cv_acc = cv_scores.mean(); cv_std = cv_scores.std()
        except Exception:
            cv_acc = 0.5; cv_std = 0.0
        probe.fit(X, y)
        train_acc = probe.score(X, y)
        probe_results.append(dict(
            layer=L, head=hi, train_acc=train_acc, cv_acc=cv_acc, cv_std=cv_std,
            probe_coef=probe.coef_[0].tolist(),
            probe_intercept=float(probe.intercept_[0])))

probe_results.sort(key=lambda r: -r['cv_acc'])

print(f'\n=== Top 20 response-time cognitive heads ===')
for r in probe_results[:20]:
    print(f'L{r["layer"]:>3d} H{r["head"]:>2d}  train {r["train_acc"]*100:>5.1f}%  cv {r["cv_acc"]*100:>5.1f}% +- {r["cv_std"]*100:.1f}')

print(f'\n=== Bottom 5 ===')
for r in probe_results[-5:]:
    print(f'L{r["layer"]:>3d} H{r["head"]:>2d}  cv {r["cv_acc"]*100:>5.1f}%')

top_k = int(len(probe_results) * 0.38)
cognitive_heads = probe_results[:top_k]
layer_counts = {}
for r in cognitive_heads:
    layer_counts[r['layer']] = layer_counts.get(r['layer'], 0) + 1
print(f'\nOK Top-{top_k} cognitive heads distribution:')
for L, c in sorted(layer_counts.items()):
    print(f'  L{L}: {c}/16')

p1_top = 57.8
p1b_top = probe_results[0]['cv_acc'] * 100
print(f'\nTop head: Phase 1 (prompt) {p1_top:.1f}% -> Phase 1b (response) {p1b_top:.1f}% (delta {p1b_top-p1_top:+.1f}pp)')

## Cell 7 — Save + upload Phase 1b

In [ ]:
output_data = {
    'ga_layers': GA_LAYERS,
    'n_q_heads': N_Q_HEADS,
    'head_dim': HEAD_DIM,
    'n_rollouts': int(len(head_acts)),
    'correct_rate': float(correct_mask.mean()),
    'capture_mode': 'response_mean_pool',
    'max_new_tokens': MAX_NEW,
    'probe_results': probe_results,
    'cognitive_heads': [{'layer': r['layer'], 'head': r['head'], 'cv_acc': r['cv_acc']}
                         for r in cognitive_heads],
    'top_k': top_k,
}
with open(OUT / 'crest_phase1b_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)

all_coefs = np.stack([np.array(r['probe_coef']) for r in probe_results]).astype(np.float32)
layer_ids = np.array([r['layer'] for r in probe_results], dtype=np.int32)
head_ids = np.array([r['head'] for r in probe_results], dtype=np.int32)
cv_accs = np.array([r['cv_acc'] for r in probe_results], dtype=np.float32)

np.savez(OUT / 'probe_coefs_response.npz',
         coefs=all_coefs, layer_ids=layer_ids, head_ids=head_ids, cv_accs=cv_accs)

api = HfApi()
for fn in ['head_acts_response.npz', 'probe_coefs_response.npz',
           'crest_phase1b_results.json']:
    try:
        api.upload_file(path_or_fileobj=str(OUT / fn),
                        path_in_repo=f'phase1b/{fn}',
                        repo_id=HF_CREST, repo_type='model',
                        commit_message=f'CREST Phase 1b: {fn}')
        print(f'OK {fn}')
    except Exception as e:
        print(f'FAIL {fn}: {e}')
print(f'\nOK uploaded to https://huggingface.co/{HF_CREST}/tree/main/phase1b')